In [ ]:
!pip install pandas numpy statsmodels scikit-learn matplotlib openpyxl xlsxwriter

In [ ]:
# !pip install pandas numpy statsmodels scikit-learn matplotlib openpyxl xlsxwriter

import pandas as pd, numpy as np, matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan, linear_reset, normal_ad
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.outliers_influence import variance_inflation_factor, OLSInfluence
from sklearn.preprocessing import StandardScaler


In [ ]:
# ==== 1) Konfigurasi ====
FILE_XLSX = "Kuesioner_SDI.xlsx"     # ganti path kalau perlu
SHEET     = "Siap Regresi"
Y_COL     = "Y_Keberhasilan"
X_COLS    = ["X1_Komunikasi","X2_SumberDaya","X3_Disposisi","X4_StrukturBirokrasi"]


In [ ]:
# ==== 2) Load data ====
df = pd.read_excel(FILE_XLSX, sheet_name=SHEET)[X_COLS+[Y_COL]].dropna().astype(float)

Y = df[Y_COL].values
X = sm.add_constant(df[X_COLS].values)
var_names = ["const"] + X_COLS

In [ ]:
# ==== 3) Fit OLS ====
model = sm.OLS(Y, X).fit()
model_hc3 = model.get_robustcov_results(cov_type='HC3')  # SE robust (HC3)

In [ ]:
# ==== 4) Uji asumsi ====
# 4a) Linieritas / spesifikasi
reset = linear_reset(model, power=2, use_f=True)

# 4b) Autokorelasi
dw = durbin_watson(model.resid)

# 4c) Homoskedastisitas
bp_stat, bp_p, f_stat, f_p = het_breuschpagan(model.resid, model.model.exog)

# 4d) Normalitas residual
ad_stat, ad_p = normal_ad(model.resid)

# 4e) Multikolinearitas (VIF) – tanpa konstanta
vif_df = pd.DataFrame({
    "Variabel": X_COLS,
    "VIF": [variance_inflation_factor(X, i+1) for i in range(len(X_COLS))]
})

# 4f) Observasi berpengaruh
infl = OLSInfluence(model)
cooks = infl.cooks_distance[0]
lev   = infl.hat_matrix_diag
n     = len(df)
k     = X.shape[1]                 # termasuk konstanta
cook_thresh = 4/n
lev_thresh  = 2*k/n

diag_df = pd.DataFrame({
    "Index": np.arange(n),
    "Leverage": lev,
    "CooksD": cooks
})
diag_df["HighCookD"] = diag_df["CooksD"] > cook_thresh
diag_df["HighLev"]   = diag_df["Leverage"] > lev_thresh

In [ ]:
# ==== 5) Koefisien (OLS & HC3) + Beta standar ====
def stars(p): return "***" if p<0.001 else "**" if p<0.01 else "*" if p<0.05 else ""
ols_tbl = pd.DataFrame({
    "Variabel": var_names,
    "B": model.params,
    "SE": model.bse,
    "t": model.tvalues,
    "p": model.pvalues,
    "Sig": [stars(p) for p in model.pvalues],
})
hc3_tbl = pd.DataFrame({
    "Variabel": var_names,
    "B": model_hc3.params,
    "SE (HC3)": model_hc3.bse,
    "t": model_hc3.tvalues,
    "p": model_hc3.pvalues,
    "Sig": [stars(p) for p in model_hc3.pvalues],
})

# Beta standar
Z = pd.DataFrame(StandardScaler().fit_transform(df), columns=df.columns)
std_model = sm.OLS(Z[Y_COL].values, sm.add_constant(Z[X_COLS].values)).fit()
beta_std = pd.DataFrame({"Variabel": X_COLS, "Beta Std": std_model.params[1:]})

In [ ]:
# ==== 6) Uji sensitivitas (drop Cook’s D tinggi) ====
drop_idx = diag_df.index[diag_df["HighCookD"]].tolist()
df2 = df.drop(df.index[drop_idx]).reset_index(drop=True)
Y2 = df2[Y_COL].values
X2 = sm.add_constant(df2[X_COLS].values)
model2 = sm.OLS(Y2, X2).fit()
sens_tbl = pd.DataFrame({
    "Variabel": var_names,
    "B (drop High-Cook)": model2.params,
    "SE": model2.bse,
    "t": model2.tvalues,
    "p": model2.pvalues,
    "Sig": [stars(p) for p in model2.pvalues],
})

In [ ]:
# ==== 7) Ringkasan & simpan ke Excel ====
summary_tbl = pd.DataFrame({
    "Metric": ["N","R2","Adj R2","F","F p","DW","BP p","AD p","RESET p","CookD thresh","Lev thresh"],
    "Value":  [int(model.nobs), model.rsquared, model.rsquared_adj, model.fvalue, model.f_pvalue,
               dw, bp_p, ad_p, reset.pvalue, cook_thresh, lev_thresh]
})
out_xlsx = "Assumption_Report_OLS.xlsx"
with pd.ExcelWriter(out_xlsx, engine="xlsxwriter") as w:
    df.to_excel(w, sheet_name="Data Used", index=False)
    summary_tbl.to_excel(w, sheet_name="Model Summary", index=False)
    ols_tbl.to_excel(w, sheet_name="Coefficients (OLS)", index=False)
    hc3_tbl.to_excel(w, sheet_name="Coefficients (HC3)", index=False)
    beta_std.to_excel(w, sheet_name="Std Betas", index=False)
    vif_df.to_excel(w, sheet_name="VIF", index=False)
    diag_df.to_excel(w, sheet_name="Influence", index=False)
    sens_tbl.to_excel(w, sheet_name="Sensitivity", index=False)

print("Saved:", out_xlsx)

Saved: Assumption_Report_OLS.xlsx


In [ ]:
!pip install -U matplotlib


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 51.7 MB/s eta 0:00:00
  Attempting uninstall: matplotlib
    Found existing installation: matplotlib 3.10.0
    Uninstalling matplotlib-3.10.0:
      Successfully uninstalled matplotlib-3.10.0


In [ ]:
# ==== 8) Grafik diagnostik (PNG) ====
# Residuals vs Fitted
plt.figure()
plt.scatter(model.fittedvalues, model.resid)
plt.axhline(0, ls="--")
plt.xlabel("Fitted")
plt.ylabel("Residuals")
plt.title("Residuals vs Fitted")
plt.tight_layout()
plt.savefig("residuals_vs_fitted.png"); plt.close()

# QQ-plot
sm.qqplot(model.resid, line="45")
plt.title("QQ Plot of Residuals")
plt.tight_layout()
plt.savefig("qq_plot_residuals.png"); plt.close()

# Cook's Distance
plt.figure()
plt.stem(np.arange(n), cooks)
plt.axhline(cook_thresh, ls="--")
plt.xlabel("Obs index")
plt.ylabel("Cook's D")
plt.title("Cook's Distance (cut-off 4/n)")
plt.tight_layout()
plt.savefig("cooks_distance.png"); plt.close()

print("Saved PNGs: residuals_vs_fitted.png, qq_plot_residuals.png, cooks_distance.png")

Saved PNGs: residuals_vs_fitted.png, qq_plot_residuals.png, cooks_distance.png


In [ ]:

print("\n== Model ==")
print(f"R2={model.rsquared:.3f} AdjR2={model.rsquared_adj:.3f}  F={model.fvalue:.2f} (p={model.f_pvalue:.3e})")
print("RESET p:", round(reset.pvalue,3), "| DW:", round(dw,2), "| BP p:", round(bp_p,3), "| AD p:", round(ad_p,3))
print("\n== VIF =="); print(vif_df)
print("\n== High influence (CookD>4/n or leverage>2k/n) ==")
print(diag_df[(diag_df.HighCookD)|(diag_df.HighLev)])



== Model ==
R2=0.871 AdjR2=0.858  F=72.27 (p=1.614e-18)
RESET p: 0.882 | DW: 2.17 | BP p: 0.124 | AD p: 0.008

== VIF ==
               Variabel       VIF
0         X1_Komunikasi  3.101788
1         X2_SumberDaya  5.545391
2          X3_Disposisi  2.096389
3  X4_StrukturBirokrasi  5.126000

== High influence (CookD>4/n or leverage>2k/n) ==
    Index  Leverage    CooksD  HighCookD  HighLev
0       0  0.136274  0.146587       True    False
4       4  0.329957  0.888465       True     True
6       6  0.251767  0.138771       True     True
15     15  0.287414  0.408047       True     True
20     20  0.115783  0.120633       True    False
45     45  0.290012  0.019705      False     True
47     47  0.260559  0.045926      False     True
